# Brain Tumor MRI Image Classification

Train and compare a custom CNN and an ImageNet-initialized MobileNetV2 using the supplied `Tumour` train/valid/test folders. Results are experimental and are not clinical evidence.

**Dataset:** 4 labels (`glioma`, `meningioma`, `no_tumor`, `pituitary`). The dataset README says 1,695/502/246; actual directory counts should be checked below (the supplied folder currently contains 1,696/503/246).


In [ ]:
from pathlib import Path
from collections import Counter
from PIL import Image

DATA = Path("Tumour")
classes = ["glioma", "meningioma", "no_tumor", "pituitary"]
for split in ("train", "valid", "test"):
    print(split, {c: len(list((DATA / split / c).glob("*.jpg"))) for c in classes})
# Review image dimensions
for c in classes:
    files=list((DATA/"train"/c).glob("*.jpg"))[:3]
    print(c, [(p.name, Image.open(p).size) for p in files])


## Custom CNN

The training utility resizes images, normalizes them with ImageNet channel statistics, augments training images, and keeps the provided validation and test splits separate. The best validation-loss checkpoint is saved to `models/cnn.pt`.


In [ ]:
from brain_tumor import train
cnn_results = train(model_kind="cnn", data_dir=DATA, epochs=6, batch_size=32, image_size=96)
cnn_results


## Transfer learning with MobileNetV2

Torchvision requests pretrained ImageNet weights. If those weights are unavailable in this environment, the utility reports the fallback and trains the same architecture from random initialization; check `pretrained_imagenet` in the results before calling this transfer learning.


In [ ]:
mobile_results = train(model_kind="mobilenet_v2", data_dir=DATA, epochs=3, batch_size=32, lr=0.0003, pretrained=True, image_size=96)
mobile_results


## Compare results

Precision, recall, F1, and the confusion matrix are computed from the held-out test split. The two JSON result files preserve metrics and training history for comparison.


In [ ]:
import json
for name in ("cnn", "mobilenet_v2"):
    path=Path("models")/f"{name}_results.json"
    if path.exists():
        r=json.loads(path.read_text())
        print(name, "ImageNet weights:", r["pretrained_imagenet"], "test accuracy:", round(r["test_accuracy"],4))
        print("per class:", r["per_class"])
        print("confusion matrix:", r["confusion_matrix"])


## Training curves and confusion matrices
Plot the saved history and test confusion matrix for each available model.

In [ ]:
import json
import matplotlib.pyplot as plt
from pathlib import Path
for name in ("cnn", "mobilenet_v2"):
    path=Path("models")/f"{name}_results.json"
    if not path.exists():
        continue
    r=json.loads(path.read_text())
    h=r["history"]
    fig,axes=plt.subplots(1,2,figsize=(10,3.5))
    axes[0].plot(h["train_accuracy"],label="train")
    axes[0].plot(h["valid_accuracy"],label="validation")
    axes[0].set(title=f"{name}: accuracy",xlabel="epoch",ylabel="accuracy"); axes[0].legend()
    axes[1].plot(h["train_loss"],label="train")
    axes[1].plot(h["valid_loss"],label="validation")
    axes[1].set(title=f"{name}: loss",xlabel="epoch",ylabel="loss"); axes[1].legend()
    plt.show()
    plt.figure(figsize=(4,4)); plt.imshow(r["confusion_matrix"],cmap="Blues")
    plt.xticks(range(4),classes,rotation=35,ha="right"); plt.yticks(range(4),classes)
    plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"{name}: test confusion matrix"); plt.colorbar(); plt.tight_layout(); plt.show()
